In [10]:
import numpy as np
from Bio import SeqIO

In [22]:
record = SeqIO.read("chr1.fasta", "fasta") #буду всё также работать с первой хромосомой человека
seq = str(record.seq).upper()
seq = seq[10000:20000] #возьмем вторые 10000 нуклеотидов

In [23]:
nucleotides = ['A', 'C', 'G', 'T']
n = len(nucleotides)
counts = np.zeros((n, n), dtype=int)

for i in range(len(seq)-1):
    if seq[i] in nucleotides and seq[i+1] in nucleotides:
        from_idx = nucleotides.index(seq[i])
        to_idx = nucleotides.index(seq[i+1])
        counts[from_idx, to_idx] += 1

print("Матрица частот динуклеотидов:")
print("     A    C    G    T")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  {counts[i,0]:4d} {counts[i,1]:4d} {counts[i,2]:4d} {counts[i,3]:4d}")

Матрица частот динуклеотидов:
     A    C    G    T
A   462  512  805  286
C   759 1090  307  872
G   570  869  986  470
T   274  557  797  383


In [25]:
P = counts / counts.sum(axis=1, keepdims=True)

print("Матрица переходов P:")
print('По строкам - направление "от" (from)')
print('По столбцам - направление "к" (to) ')
print("     A      C      G      T")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}  {P[i,0]:.4f} {P[i,1]:.4f} {P[i,2]:.4f} {P[i,3]:.4f}")

Матрица переходов P:
По строкам - направление "от" (from)
По столбцам - направление "к" (to) 
     A      C      G      T
A  0.2237 0.2479 0.3898 0.1385
C  0.2507 0.3600 0.1014 0.2880
G  0.1969 0.3002 0.3406 0.1623
T  0.1363 0.2770 0.3963 0.1905


In [26]:
print("Сумма элементов в каждой строке:")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}: {P[i].sum():.4f}")

Сумма элементов в каждой строке:
A: 1.0000
C: 1.0000
G: 1.0000
T: 1.0000


Найдем стационарное распр-е, решив ур-е $\pi = \pi P$:

In [27]:
eigenvals, eigenvecs = np.linalg.eig(P.T)
stationary = eigenvecs[:, np.isclose(eigenvals, 1)].real
stationary = stationary / stationary.sum()
stationary = stationary.flatten()

print("Стационарное распределение π:")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}: {stationary[i]:.6f}")

Стационарное распределение π:
A: 0.206521
C: 0.302830
G: 0.289529
T: 0.201120


Сравним с наблюдаемыми частотами:

In [28]:
obs_freq = np.array([seq.count(nuc) for nuc in nucleotides]) / len(seq)

print("Наблюдаемые частоты нуклеотидов:")
for i, nuc in enumerate(nucleotides):
    print(f"{nuc}: {obs_freq[i]:.6f}")

Наблюдаемые частоты нуклеотидов:
A: 0.206500
C: 0.302800
G: 0.289500
T: 0.201200


Как можно наблюдать, частоты стационарного распределения в точности совпадают с наблюдаемыми частотами отдельных нуклеотидов. По кайней мере с точностью до 4-го знака.